<a href="https://colab.research.google.com/github/pradervonsky/vbig-lab/blob/main/evaluation/distance-based_IAA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inter-Annotator Agreement (IAA)

Pipeline:
1. Pull `human_insights` rows from Supabase
2. Build units, parsing the dashboard-chart-level LoD
3. Compute BERTScore-based pairwise distances for all 3 annotator pairs
4. Compute Krippendorff's α overall, per level, and per dashboard
5. Compute ROUGE, BLEU, METEOR, BERTScore

## Initial steps

In [1]:
!pip install -q bert-score supabase pandas numpy openpyxl rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
import torch
import nltk
from itertools import combinations
from bert_score import BERTScorer
from bert_score import score as bert_score
from supabase import create_client
from google.colab import userdata
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

### Pull data from Supabase

In [4]:
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
response = supabase.table("human_insights").select("*").execute()
df_raw = pd.DataFrame(response.data)

print("irr_flag unique values:", df_raw["irr_flag"].unique())
irr_df = df_raw[df_raw["irr_flag"] == True].reset_index(drop=True)
print(f"IRR rows: {len(irr_df)}")

irr_flag unique values: [False  True]
IRR rows: 10


### Parsing step

In [5]:
NA_PATTERN = re.compile(r"^(not applicable|n/a)$", re.IGNORECASE)

def is_missing(text):
    if text is None:
        return True
    # Strip whitespace and trailing punctuation before matching
    cleaned = str(text).strip().rstrip(".")
    cleaned = cleaned.strip()  # strip again after removing dot
    return cleaned.lower() in {"not applicable", "n/a", ""}

def normalize_value(val):
    if val is None:
        return None
    cleaned = str(val).strip().rstrip(".").strip()
    if cleaned == "" or NA_PATTERN.match(cleaned):
        return None
    return cleaned

def parse_charts(text):
    charts = []
    if not text or (isinstance(text, float) and np.isnan(text)):
        return charts

    blocks = re.split(r"(?=Chart\s+\d+\s*[:.])", text.strip())
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        lines = block.splitlines()
        header = lines[0].strip()
        match = re.match(r"Chart\s+(\d+)\s*[:.]\s*(.*)", header)
        if not match:
            continue
        chart_id = int(match.group(1))
        title    = match.group(2).strip()

        L2 = L3 = L4 = None
        for line in lines[1:]:
            line = line.strip()
            if line.startswith("L2:"):
                L2 = normalize_value(line[3:].strip())
            elif line.startswith("L3:"):
                L3 = normalize_value(line[3:].strip())
            elif line.startswith("L4:"):
                L4 = normalize_value(line[3:].strip())

        charts.append({
            "chart_id": chart_id,
            "title":    title,
            "L2":       L2,
            "L3":       L3,
            "L4":       L4,
        })
    return charts

In [6]:
ANNOTATOR_COLS = ["insight_part_1", "insight_part_2", "insight_part_3"]

records = []
for _, row in irr_df.iterrows():
    row_id      = row["id"]
    metadata_id = row["metadata_id"]

    # Parse each annotator's blob
    parsed = {col: {c["chart_id"]: c for c in parse_charts(row[col])}
              for col in ANNOTATOR_COLS}

    # All chart_ids seen across any annotator
    all_chart_ids = sorted(
        set().union(*[set(p.keys()) for p in parsed.values()])
    )

    for chart_id in all_chart_ids:
        # Get title from whichever annotator has this chart
        title = next(
            (parsed[col][chart_id]["title"]
             for col in ANNOTATOR_COLS
             if chart_id in parsed[col]),
            ""
        )
        for level in ["L2", "L3", "L4"]:
            records.append({
                "row_id":      row_id,
                "metadata_id": metadata_id,
                "chart_id":    chart_id,
                "title":       title,
                "level":       level,
                "unit_id":     f"{row_id}__chart{chart_id}__{level}",
                "annotator_1": parsed["insight_part_1"].get(chart_id, {}).get(level),
                "annotator_2": parsed["insight_part_2"].get(chart_id, {}).get(level),
                "annotator_3": parsed["insight_part_3"].get(chart_id, {}).get(level),
            })

long_df = pd.DataFrame(records)
print(f"\nTotal units: {len(long_df)}")
print(long_df.head(12).to_string())


Total units: 147
                                  row_id                           metadata_id  chart_id                        title level                                           unit_id                                                                                                                                                                                                                                                                                             annotator_1                                                                                                                                                                                                                                                                                                              annotator_2                                                                                                                                                                                                        

In [7]:
print("\nNOT APPLICABLE counts per level per annotator:")
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    print(f"  {level}: "
          f"ann1={sub['annotator_1'].isna().sum()} | "
          f"ann2={sub['annotator_2'].isna().sum()} | "
          f"ann3={sub['annotator_3'].isna().sum()} | "
          f"total_units={len(sub)}")


NOT APPLICABLE counts per level per annotator:
  L2: ann1=0 | ann2=0 | ann3=0 | total_units=49
  L3: ann1=14 | ann2=4 | ann3=2 | total_units=49
  L4: ann1=0 | ann2=0 | ann3=0 | total_units=49


In [8]:
# Check chart count per dashboard per annotator
print("=== Chart count per dashboard per annotator ===\n")

for row_id in sorted(long_df["row_id"].unique()):
    sub = long_df[long_df["row_id"] == row_id]

    # Get unique charts seen per annotator
    ann1_charts = sub[sub["annotator_1"].notna()]["chart_id"].unique()
    ann2_charts = sub[sub["annotator_2"].notna()]["chart_id"].unique()
    ann3_charts = sub[sub["annotator_3"].notna()]["chart_id"].unique()

    total_charts = sub["chart_id"].nunique()
    metadata_id = sub["metadata_id"].iloc[0]

    print(f"Dashboard: {row_id[:8]}… (metadata: {metadata_id[:8]}…)")
    print(f"  Total charts parsed: {total_charts}")
    print(f"  ann1 charts with content: {sorted(ann1_charts)} ({len(ann1_charts)})")
    print(f"  ann2 charts with content: {sorted(ann2_charts)} ({len(ann2_charts)})")
    print(f"  ann3 charts with content: {sorted(ann3_charts)} ({len(ann3_charts)})")

    # Flag mismatches
    all_charts = set(sub["chart_id"].unique())
    for ann_label, ann_charts in [("ann1", ann1_charts), ("ann2", ann2_charts), ("ann3", ann3_charts)]:
        missing = all_charts - set(ann_charts)
        if missing:
            print(f"  ⚠️  {ann_label} missing charts: {sorted(missing)}")
    print()

=== Chart count per dashboard per annotator ===

Dashboard: 08006d53… (metadata: 27052e58…)
  Total charts parsed: 4
  ann1 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4)
  ann2 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4)
  ann3 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4)

Dashboard: 1120a289… (metadata: 4f4b551b…)
  Total charts parsed: 5
  ann1 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] (5)
  ann2 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] (5)
  ann3 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] (5)

Dashboard: 2e5b2881… (metadata: 932c11c0…)
  Total charts parsed: 6
  ann1 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)] (6)
  ann2 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.i

### BERTScore-based distance function

`distance(a, b) = 1 - BERTScore_F1(a, b)`  
`NOT APPLICABLE` entries are treated as missing (`np.nan`).

In [9]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_TYPE = "roberta-large"
print(f"Using device: {DEVICE}, model: {MODEL_TYPE}")

SCORER = BERTScorer(
    model_type=MODEL_TYPE,
    lang="en",
    device=DEVICE
)

ANNOTATOR_COLS = ["annotator_1", "annotator_2", "annotator_3"]
ANNOTATOR_PAIRS = list(combinations(ANNOTATOR_COLS, 2))

Using device: cuda, model: roberta-large


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def is_missing(text):
    if text is None:
        return True
    return str(text).strip().lower() in {"not applicable", "n/a", ""}

def bertscore_distance_batch(refs, hyps):
    assert len(refs) == len(hyps)
    distances = np.full(len(refs), np.nan)

    valid_indices = [
        i for i, (r, h) in enumerate(zip(refs, hyps))
        if not is_missing(r) and not is_missing(h)
    ]
    if not valid_indices:
        return distances

    valid_refs = [str(refs[i]) for i in valid_indices]
    valid_hyps = [str(hyps[i]) for i in valid_indices]

    _, _, F1 = SCORER.score(valid_hyps, valid_refs)

    for idx, f1_val in zip(valid_indices, F1.cpu().numpy()):
        # Clamp F1 to [0,1] before converting to distance
        # rescaled BERTScore can be negative for very dissimilar pairs
        f1_clamped = float(np.clip(f1_val, 0.0, 1.0))
        distances[idx] = 1.0 - f1_clamped

    return distances

In [11]:
# Compute pairwise distances on long_df
DIST_COLS = []
for (col_a, col_b) in ANNOTATOR_PAIRS:
    suffix_a = col_a.split("_")[-1]
    suffix_b = col_b.split("_")[-1]
    pair_key = f"dist_{suffix_a}{suffix_b}"
    DIST_COLS.append(pair_key)
    print(f"Computing {pair_key} ({col_a} vs {col_b}) ...")
    long_df[pair_key] = bertscore_distance_batch(
        long_df[col_a].tolist(),
        long_df[col_b].tolist(),
    )

print("\nSample distances:")
print(long_df[["unit_id", "chart_id", "level"] + DIST_COLS].head(12).to_string(index=False))

Computing dist_12 (annotator_1 vs annotator_2) ...
Computing dist_13 (annotator_1 vs annotator_3) ...
Computing dist_23 (annotator_2 vs annotator_3) ...

Sample distances:
                                         unit_id  chart_id level  dist_12  dist_13  dist_23
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L2         1    L2 0.059479 0.077053 0.056197
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L3         1    L3      NaN      NaN      NaN
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L4         1    L4 0.129179 0.135945 0.126873
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L2         2    L2 0.095340 0.127698 0.092421
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L3         2    L3 0.131507 0.141471 0.125049
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L4         2    L4 0.130863 0.139733 0.103228
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart3__L2         3    L2 0.075207 0.083880 0.065052
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart3__L3         3    L3 0.113063 0.138876 0.116854


## Distance-based Inter-Annotator Agreement Framework

### Distributional Agreement Metrics (Sigma and KS)

> This section introduces distributional agreement metrics following Braylan et al. (2022), where agreement is evaluated based on the separation between observed and expected disagreement distributions.

> The sigma (\(\sigma\)) metric quantifies the proportion of observed annotation pairs that are significantly more similar than random expectation, while the Kolmogorov–Smirnov (KS) statistic measures the degree of separation between the observed and expected disagreement distributions.

In [18]:
from scipy.stats import ks_2samp
import numpy as np
import pandas as pd

def get_observed_distances(sub_df, dist_cols=DIST_COLS):
    """
    Do: pairwise annotator distances within the same unit.
    Uses existing dist_12, dist_13, dist_23 columns.
    """
    vals = sub_df[dist_cols].values.flatten()
    vals = vals[~np.isnan(vals)]
    return vals


def get_expected_distances(sub_df, max_pairs=50000, random_state=42):
    """
    De: pairwise distances between annotations from different units.
    This pools all annotator outputs, then compares only cross-unit pairs.
    Sampling is used to avoid very slow all-vs-all computation.
    """
    rng = np.random.default_rng(random_state)

    pool = []
    for _, row in sub_df.iterrows():
        for ann_col in ["annotator_1", "annotator_2", "annotator_3"]:
            txt = row[ann_col]
            if not is_missing(txt):
                pool.append({
                    "unit_id": row["unit_id"],
                    "text": str(txt)
                })

    if len(pool) < 2:
        return np.array([])

    pairs = []
    attempts = 0
    max_attempts = max_pairs * 10

    while len(pairs) < max_pairs and attempts < max_attempts:
        i, j = rng.choice(len(pool), size=2, replace=False)
        attempts += 1

        if pool[i]["unit_id"] == pool[j]["unit_id"]:
            continue

        pairs.append((pool[i]["text"], pool[j]["text"]))

    if not pairs:
        return np.array([])

    refs = [p[0] for p in pairs]
    hyps = [p[1] for p in pairs]

    _, _, F1 = SCORER.score(hyps, refs)
    F1_clamped = np.clip(F1.cpu().numpy(), 0.0, 1.0)

    return 1.0 - F1_clamped


def compute_sigma_and_ks(sub_df, p_threshold=0.05, max_expected_pairs=50000):
    """
    Braylan-style distributional IAA:
    - KS is used to compare separation between Do and De.
    - sigma is the proportion of observed distances significantly smaller than chance.
    """
    Do = get_observed_distances(sub_df)
    De = get_expected_distances(sub_df, max_pairs=max_expected_pairs)

    if len(Do) == 0 or len(De) == 0:
        return {
            "sigma": np.nan,
            "ks_score": np.nan,
            "ks_pvalue": np.nan,
            "Do_mean": np.nan,
            "De_mean": np.nan,
            "n_Do": len(Do),
            "n_De": len(De)
        }

    # empirical CDF of De at each observed distance d
    De_sorted = np.sort(De)
    cdf_values = np.searchsorted(De_sorted, Do, side="right") / len(De_sorted)

    sigma = np.mean(cdf_values < p_threshold)

    ks_stat, ks_pvalue = ks_2samp(Do, De, alternative="greater")
    ks_score = ks_stat

    return {
        "sigma": sigma,
        "ks_score": ks_score,
        "ks_pvalue": ks_pvalue,
        "Do_mean": np.mean(Do),
        "De_mean": np.mean(De),
        "n_Do": len(Do),
        "n_De": len(De)
    }

In [19]:
sigma_results = []

for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]

    print(f"\nComputing sigma for {level}...")
    result = compute_sigma_and_ks(sub)

    result["Level"] = level
    sigma_results.append(result)

sigma_df = pd.DataFrame(sigma_results)

sigma_df = sigma_df[
    ["Level", "sigma", "ks_score", "ks_pvalue", "Do_mean", "De_mean", "n_Do", "n_De"]
]

print(sigma_df.to_string(index=False))


Computing sigma for L2...

Computing sigma for L3...

Computing sigma for L4...
Level    sigma  ks_score    ks_pvalue  Do_mean  De_mean  n_Do  n_De
   L2 0.687075  0.691466 8.511021e-62 0.097185 0.153327   147 50000
   L3 0.278261  0.493893 3.523219e-25 0.129602 0.150843   115 50000
   L4 0.190476  0.371479 2.108545e-18 0.134846 0.145765   147 50000


### Krippendorff's alpha with BERTScore distance

>Krippendorff’s alpha is computed using BERTScore-based distances to provide a baseline agreement measure. However, given known limitations of α for complex annotation tasks, it is reported for completeness and interpreted alongside distributional agreement metrics.

In [14]:
def krippendorff_alpha_bertscore_pooled(sub_df, dist_cols=DIST_COLS, max_expected_pairs=50000, random_state=42):
    # Observed disagreement: all within-unit annotator-pair distances
    Do_vals = sub_df[dist_cols].values.flatten()
    Do_vals = Do_vals[~np.isnan(Do_vals)]

    if len(Do_vals) == 0:
        return np.nan, np.nan, np.nan, 0

    D_o = np.mean(Do_vals)

    # Expected disagreement: pooled annotations from different units
    pool = []
    for _, row in sub_df.iterrows():
        for ann_col in ["annotator_1", "annotator_2", "annotator_3"]:
            txt = row[ann_col]
            if not is_missing(txt):
                pool.append({
                    "unit_id": row["unit_id"],
                    "text": str(txt)
                })

    if len(pool) < 2:
        return np.nan, D_o, np.nan, len(Do_vals)

    rng = np.random.default_rng(random_state)
    pairs = []
    attempts = 0
    max_attempts = max_expected_pairs * 10

    while len(pairs) < max_expected_pairs and attempts < max_attempts:
        i, j = rng.choice(len(pool), size=2, replace=False)
        attempts += 1

        if pool[i]["unit_id"] == pool[j]["unit_id"]:
            continue

        pairs.append((pool[i]["text"], pool[j]["text"]))

    refs = [p[0] for p in pairs]
    hyps = [p[1] for p in pairs]

    _, _, F1 = SCORER.score(hyps, refs)
    F1_clamped = np.clip(F1.cpu().numpy(), 0.0, 1.0)

    De_vals = 1.0 - F1_clamped
    D_e = np.mean(De_vals)

    alpha = 1.0 - (D_o / D_e)

    return alpha, D_o, D_e, len(Do_vals)

In [21]:
level_results = []

for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    alpha, D_o, D_e, n_obs = krippendorff_alpha_bertscore_pooled(sub)

    level_results.append({
        "Level": level,
        "Alpha": alpha,
        "D_o": D_o,
        "D_e": D_e,
        "N_observed_pairs": n_obs
    })

alpha_df = pd.DataFrame(level_results)
print(alpha_df)

  Level     Alpha       D_o       D_e  N_observed_pairs
0    L2  0.366155  0.097185  0.153327               147
1    L3  0.140818  0.129602  0.150843               115
2    L4  0.074910  0.134846  0.145765               147


### IAA Results

In [22]:
final_df = alpha_df.merge(sigma_df, on="Level")

final_df = final_df[[
    "Level",
    "Alpha",
    "sigma",
    "ks_score",
    "D_o",
    "D_e"
]]

print(final_df.to_string(index=False))

Level    Alpha    sigma  ks_score      D_o      D_e
   L2 0.366155 0.687075  0.691466 0.097185 0.153327
   L3 0.140818 0.278261  0.493893 0.129602 0.150843
   L4 0.074910 0.190476  0.371479 0.134846 0.145765


In [24]:
def interpret_sigma(s):
    if s >= 0.6:
        return "Moderate non-random agreement"
    elif s >= 0.25:
        return "Low agreement (interpretive)"
    else:
        return "Very low agreement (subjective)"

final_df["Interpretation"] = final_df["sigma"].apply(interpret_sigma)
print(final_df)

  Level     Alpha     sigma  ks_score       D_o       D_e  \
0    L2  0.366155  0.687075  0.691466  0.097185  0.153327   
1    L3  0.140818  0.278261  0.493893  0.129602  0.150843   
2    L4  0.074910  0.190476  0.371479  0.134846  0.145765   

                    Interpretation  
0    Moderate non-random agreement  
1     Low agreement (interpretive)  
2  Very low agreement (subjective)  
